# 06 - Model Explainability
SHAP analysis, priority banding, churn reason analysis, and customer risk table output.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import shap
%matplotlib inline

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from config import (PROCESSED_DATA_DIR, MODELS_DIR, OUTPUT_DIR,
                    PRIORITY_BANDS, RETENTION_PLAYBOOK, RANDOM_STATE, TEST_SIZE)
from src.util import load_dataframe, load_model, save_dataframe

## 6.1 Load data and best model

In [ ]:
df = load_dataframe(os.path.join(PROCESSED_DATA_DIR, 'df_engineered.csv'))

# load training artifacts
artifacts = joblib.load(os.path.join(PROCESSED_DATA_DIR, 'training_artifacts.pkl'))
selected = artifacts['selected_features']
best_name = artifacts['best_model_name']

# load best model
safe_name = best_name.lower().replace(' ', '_')
best_model = load_model(os.path.join(MODELS_DIR, f'{safe_name}.pkl'))

print(f'Best model: {best_name}')
print(f'Features: {len(selected)}')

In [ ]:
# prepare data
X = df[selected].copy().fillna(0).replace([np.inf, -np.inf], 0)
y = df['is_churned'].copy()

# for SHAP, use a tree-based model (not logistic regression)
# if best model is LR, load Random Forest for SHAP instead
if best_name == 'Logistic Regression':
    shap_model = load_model(os.path.join(MODELS_DIR, 'random_forest.pkl'))
    shap_model_name = 'Random Forest'
    print('Using Random Forest for SHAP (tree-based explainer)')
else:
    shap_model = best_model
    shap_model_name = best_name

## 6.2 SHAP analysis

In [ ]:
# use a sample for speed (SHAP can be slow on large datasets)
sample_size = min(2000, len(X))
X_sample = X.sample(n=sample_size, random_state=RANDOM_STATE)

# create SHAP explainer
explainer = shap.TreeExplainer(shap_model)
shap_values = explainer.shap_values(X_sample)

# for binary classification, use class 1 (churned)
if isinstance(shap_values, list):
    shap_vals = shap_values[1]
else:
    shap_vals = shap_values

print(f'SHAP values computed for {sample_size:,} samples')

In [ ]:
# SHAP summary bar plot (top 15 features)
plt.figure(figsize=(10, 7))
shap.summary_plot(shap_vals, X_sample, plot_type='bar', max_display=15, show=False)
plt.title(f'SHAP Feature Importance ({shap_model_name})')
plt.tight_layout()
plt.show()

In [ ]:
# SHAP beeswarm plot
plt.figure(figsize=(10, 7))
shap.summary_plot(shap_vals, X_sample, max_display=15, show=False)
plt.title(f'SHAP Beeswarm Plot ({shap_model_name})')
plt.tight_layout()
plt.show()

In [ ]:
# top churn drivers table
mean_abs_shap = np.abs(shap_vals).mean(axis=0)
shap_importance = pd.DataFrame({
    'Feature': selected,
    'Mean |SHAP|': mean_abs_shap
}).sort_values('Mean |SHAP|', ascending=False).reset_index(drop=True)

print('Top 10 Churn Drivers (by SHAP):')
for i, row in shap_importance.head(10).iterrows():
    print(f'  {i+1:2d}. {row["Feature"]:40s} {row["Mean |SHAP|"]:.4f}')

## 6.3 Priority banding

In [ ]:
# score all cases using best model
if best_name == 'Logistic Regression':
    scaler = StandardScaler()
    # fit on same train split
    X_train, _, _, _ = train_test_split(X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y)
    scaler.fit(X_train)
    X_full_scaled = pd.DataFrame(scaler.transform(X), columns=selected, index=X.index)
    df['churn_probability'] = best_model.predict_proba(X_full_scaled)[:, 1]
else:
    df['churn_probability'] = best_model.predict_proba(X)[:, 1]

# assign priority bands
def assign_band(prob):
    for band, (low, high) in PRIORITY_BANDS.items():
        if low <= prob < high:
            return band
    return 'P4 - Low'

df['priority_band'] = df['churn_probability'].apply(assign_band)
df['revenue_at_risk'] = df['VAN'] * df['churn_probability']

print('Priority band distribution:')
print(df['priority_band'].value_counts().sort_index())

In [ ]:
# priority band summary
print(f'{"Band":<20} {"Count":>8} {"Revenue@Risk":>15} {"Avg Prob":>10} {"Actual Churn":>12}')
print('-' * 70)

for band in PRIORITY_BANDS:
    mask = df['priority_band'] == band
    if mask.sum() > 0:
        count = mask.sum()
        rev = df.loc[mask, 'revenue_at_risk'].sum()
        avg_prob = df.loc[mask, 'churn_probability'].mean()
        actual = df.loc[mask, 'is_churned'].mean()
        print(f'{band:<20} {count:>8,} {"\u00a3":>1}{rev:>13,.0f} {avg_prob:>9.1%} {actual:>11.1%}')

In [ ]:
# priority band visualizations
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

band_colors = {'P1 - Critical': '#e74c3c', 'P2 - High': '#e67e22',
               'P3 - Medium': '#f1c40f', 'P4 - Low': '#2ecc71'}

# count by band
band_counts = df['priority_band'].value_counts().reindex(PRIORITY_BANDS.keys())
axes[0].bar(range(len(band_counts)), band_counts.values,
            color=[band_colors[b] for b in band_counts.index], edgecolor='black')
axes[0].set_xticks(range(len(band_counts)))
axes[0].set_xticklabels([b.split(' - ')[0] for b in band_counts.index])
axes[0].set_title('Cases by Priority Band')
axes[0].set_ylabel('Count')

# revenue at risk by band
band_rev = df.groupby('priority_band')['revenue_at_risk'].sum().reindex(PRIORITY_BANDS.keys())
axes[1].bar(range(len(band_rev)), band_rev.values,
            color=[band_colors[b] for b in band_rev.index], edgecolor='black')
axes[1].set_xticks(range(len(band_rev)))
axes[1].set_xticklabels([b.split(' - ')[0] for b in band_rev.index])
axes[1].set_title('Revenue at Risk by Priority Band')
axes[1].set_ylabel('Revenue (\u00a3)')

# actual churn rate by band
band_churn = df.groupby('priority_band')['is_churned'].mean().reindex(PRIORITY_BANDS.keys()) * 100
axes[2].bar(range(len(band_churn)), band_churn.values,
            color=[band_colors[b] for b in band_churn.index], edgecolor='black')
axes[2].set_xticks(range(len(band_churn)))
axes[2].set_xticklabels([b.split(' - ')[0] for b in band_churn.index])
axes[2].set_title('Actual Churn Rate by Priority Band')
axes[2].set_ylabel('Churn Rate (%)')

plt.tight_layout()
plt.show()

## 6.4 Churn reason analysis

In [ ]:
if 'churn_reason_group' in df.columns:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # churn reason distribution
    reason_counts = df[df['is_churned'] == 1]['churn_reason_group'].value_counts()
    reason_counts.plot(kind='bar', ax=axes[0], color='#e74c3c', edgecolor='black', alpha=0.8)
    axes[0].set_title('Churn Reasons (Lost Customers Only)')
    axes[0].set_ylabel('Count')
    
    # revenue lost by reason
    churned = df[df['is_churned'] == 1]
    rev_by_reason = churned.groupby('churn_reason_group')['VAN'].sum().sort_values(ascending=False)
    rev_by_reason.plot(kind='bar', ax=axes[1], color='#c0392b', edgecolor='black', alpha=0.8)
    axes[1].set_title('Revenue Lost by Churn Reason')
    axes[1].set_ylabel('VAN (\u00a3)')
    
    plt.tight_layout()
    plt.show()

## 6.5 Retention playbook

In [ ]:
print('\nRetention Playbook:')
print('=' * 80)
for reason, details in RETENTION_PLAYBOOK.items():
    # count cases in our data
    if 'churn_reason_group' in df.columns:
        n = len(df[(df['churn_reason_group'] == reason) & (df['is_churned'] == 1)])
    else:
        n = 0
    print(f'\n  Reason: {reason} ({n} churned cases)')
    print(f'  Action: {details["action"]}')
    print(f'  Urgency: {details["urgency"]}')

## 6.6 Export customer risk table

In [ ]:
# build risk table
cols = ['Case ID', 'Customer Account Number', 'CompanySize', 'Customer Tier',
        'VAN', 'Machines', 'Number of Contracts', 'Case Type',
        'churn_reason_group', 'churn_probability', 'priority_band', 'revenue_at_risk', 'is_churned']
available = [c for c in cols if c in df.columns]

risk_table = df[available].copy()
risk_table = risk_table.rename(columns={
    'Case ID': 'Case_ID',
    'Customer Account Number': 'Account_Number',
    'CompanySize': 'Company_Size',
    'Customer Tier': 'Customer_Tier',
    'Number of Contracts': 'Contracts',
    'Case Type': 'Case_Type',
    'churn_reason_group': 'Churn_Reason',
    'churn_probability': 'Churn_Probability',
    'priority_band': 'Priority_Band',
    'revenue_at_risk': 'Revenue_at_Risk',
    'is_churned': 'Actual_Churned',
})

# add recommended actions
risk_table['Recommended_Action'] = risk_table['Churn_Reason'].map(
    {k: v['action'] for k, v in RETENTION_PLAYBOOK.items()}
)

# sort by priority
priority_order = {'P1 - Critical': 0, 'P2 - High': 1, 'P3 - Medium': 2, 'P4 - Low': 3}
risk_table['_sort'] = risk_table['Priority_Band'].map(priority_order)
risk_table = risk_table.sort_values(['_sort', 'Revenue_at_Risk'], ascending=[True, False])
risk_table = risk_table.drop('_sort', axis=1)

# save to output folder
os.makedirs(OUTPUT_DIR, exist_ok=True)
out_path = os.path.join(OUTPUT_DIR, 'customer_risk_table.csv')
risk_table.to_csv(out_path, index=False)
print(f'Risk table saved: {out_path}')
print(f'Total rows: {len(risk_table):,}')

# show top 10 highest risk
print('\nTop 10 highest risk customers:')
display(risk_table.head(10))